In [17]:
import json
import sys
from pathlib import Path

# Resolve the repository root whether the notebook is launched from the repo root
# or from inside the Notebooks directory.
repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "test_001_nvda").exists():
    repo_root = repo_root.parent

# Make the shared research package importable inside the notebook kernel.
sys.path.insert(0, str(repo_root))

import common


# Keep the core trading gate in this notebook because it defines the actual
# strategy rule: MACD crossover confirmed by RSI.
def add_gate_signals(df):
    df = df.copy()
    df["buy_signal"] = (
        (df["macd_line"] > df["macd_signal"])
        & (df["macd_line"].shift(1) <= df["macd_signal"].shift(1))
        & (df["rsi"] > 50)
    )
    df["sell_signal"] = (
        (df["macd_line"] < df["macd_signal"])
        & (df["macd_line"].shift(1) >= df["macd_signal"].shift(1))
        & (df["rsi"] < 50)
    )
    return df


# Load the strategy configuration once so every downstream cell uses the same
# symbol, date range, model name, and data provider settings.
STRATEGY_DIR = repo_root / "backtesting" / "test_001_nvda"
config = common.load_strategy_config(STRATEGY_DIR / "strategy_config.json")

print(f"Strategy directory: {STRATEGY_DIR}")
print(f"Model name: {config['model_name']}")
print(f"Symbol: {config['stock_symbol']}")
print(f"Backtest window: {config['backtest_start']} -> {config['backtest_end']}")

Strategy directory: /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda
Model name: test-001-nvda
Symbol: NVDA
Backtest window: 2022-01-01 -> 2023-01-01


In [18]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# ── Strategy-local definitions ───────────────────────────────────────────────
# These encode the state space and reward model for this specific strategy.
# They live here (not in common/) because they are strategy-essential: changing
# the state columns, reward logic, or gating rules requires editing this notebook.

STATE_COLUMNS = ["macd_line", "macd_signal", "macd_hist", "rsi", "cci", "position_flag"]


def build_transitions(df, transaction_cost=5e-4):
    """Build state-transition records for FQI from an indicator-enriched OHLCV DataFrame."""
    df = df.copy()
    df["next_return"] = df["close"].pct_change().shift(-1)
    df = df.iloc[:-1].copy()

    records = []
    for row_number in range(len(df) - 1):
        current_row = df.iloc[row_number]
        next_row = df.iloc[row_number + 1]
        next_return = current_row["next_return"]

        for position_flag in (0, 1):
            valid_actions = ["hold"]
            if position_flag == 0 and bool(current_row["buy_signal"]):
                valid_actions.append("buy")
            if position_flag == 1 and bool(current_row["sell_signal"]):
                valid_actions.append("sell")

            for action_name in valid_actions:
                next_position_flag = position_flag
                if action_name == "buy":
                    next_position_flag = 1
                    reward = next_return - transaction_cost
                elif action_name == "hold":
                    reward = next_return if position_flag == 1 else 0.0
                else:  # sell
                    next_position_flag = 0
                    reward = -transaction_cost

                record = {
                    "state_row": row_number,
                    "date": str(df.index[row_number]),
                    "action_name": action_name,
                    "reward": reward,
                    "buy_signal": int(bool(current_row["buy_signal"])),
                    "sell_signal": int(bool(current_row["sell_signal"])),
                    "next_buy_signal": int(bool(next_row["buy_signal"])),
                    "next_sell_signal": int(bool(next_row["sell_signal"])),
                    "next_position_flag": next_position_flag,
                }
                for col in STATE_COLUMNS:
                    if col == "position_flag":
                        record[col] = position_flag
                        record[f"next_{col}"] = next_position_flag
                    else:
                        record[col] = current_row[col]
                        record[f"next_{col}"] = next_row[col]

                records.append(record)

    return pd.DataFrame(records)


_DEFAULT_XGB_PARAMS = dict(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    objective="reg:squarederror",
)


def score_valid_actions(feature_frame, buy_signal, sell_signal, position_flag, models):
    """Score all valid actions for each state. Invalid actions receive -inf."""
    zero = np.zeros(len(feature_frame))
    scores = pd.DataFrame(index=feature_frame.index)

    hold_model = models.get("hold")
    buy_model = models.get("buy")
    sell_model = models.get("sell")

    scores["hold"] = zero if hold_model is None else hold_model.predict(feature_frame)
    buy_scores = zero if buy_model is None else buy_model.predict(feature_frame)
    sell_scores = zero if sell_model is None else sell_model.predict(feature_frame)

    scores["buy"] = np.where((position_flag == 0) & (buy_signal == 1), buy_scores, -np.inf)
    scores["sell"] = np.where((position_flag == 1) & (sell_signal == 1), sell_scores, -np.inf)
    return scores


def fitted_q_iteration(transitions, n_iter=8, gamma=0.95, xgb_params=None):
    """Run Fitted Q-Iteration and return a dict of trained XGBRegressor models."""
    if xgb_params is None:
        xgb_params = _DEFAULT_XGB_PARAMS

    action_names = ["hold", "buy", "sell"]
    models = {name: None for name in action_names}
    targets = transitions["reward"].copy()

    for _ in range(n_iter):
        next_features = transitions[[f"next_{c}" for c in STATE_COLUMNS]].copy()
        next_features.columns = STATE_COLUMNS

        next_scores = score_valid_actions(
            next_features,
            transitions["next_buy_signal"],
            transitions["next_sell_signal"],
            transitions["next_position_flag"],
            models,
        )
        max_next_q = next_scores.max(axis=1).replace(-np.inf, 0.0)
        targets = transitions["reward"] + gamma * max_next_q

        for action_name in action_names:
            action_slice = transitions[transitions["action_name"] == action_name]
            if action_slice.empty:
                continue
            model = XGBRegressor(**xgb_params)
            model.fit(action_slice[STATE_COLUMNS], targets.loc[action_slice.index])
            models[action_name] = model

    return models

In [19]:
# FEE_BPS models transaction cost in basis points. GAMMA is the discount factor
# used by fitted Q-iteration when estimating long-term action value.
FEE_BPS, GAMMA = 5, 0.95

# Pull the raw OHLCV data for the configured research window.
df = common.fetch_ohlcv(
    config["stock_symbol"],
    start=config["backtest_start"],
    end=config["backtest_end"],
    provider=config["data_src"],
)

# Build indicators first, then apply the notebook-local strategy gate.
# The final dropna() removes the warm-up rows created by rolling indicators.
df = common.add_indicators(df).pipe(add_gate_signals).dropna()

# Convert the price history into state/action/reward transitions for RL training.
transitions = build_transitions(df, transaction_cost=FEE_BPS / 10_000)

print(f"Bars: {len(df)}  ({df.index[0]} -> {df.index[-1]})")
transitions["action_name"].value_counts()

Bars: 232  (2022-01-31 -> 2022-12-30)


action_name
hold    460
sell      7
buy       4
Name: count, dtype: int64

In [20]:
# Train one Q-model per action using fitted Q-iteration.
action_models = fitted_q_iteration(transitions, n_iter=8, gamma=GAMMA)

# Build a compact catalog of unique states so we can inspect the policy's chosen
# action after training. This is useful for sanity-checking, not for export.
state_catalog = (
    transitions[["state_row", "date", *STATE_COLUMNS, "buy_signal", "sell_signal"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Score every valid action for each state. Invalid actions are masked internally
# by score_valid_actions and receive -inf so they cannot be selected.
scores = score_valid_actions(
    state_catalog[STATE_COLUMNS],
    state_catalog["buy_signal"],
    state_catalog["sell_signal"],
    state_catalog["position_flag"],
    action_models,
)
state_catalog["chosen_action"] = scores.idxmax(axis=1)

state_catalog[
    ["date", "position_flag", "buy_signal", "sell_signal", "chosen_action"]
].head()

,date,position_flag,buy_signal,sell_signal,chosen_action
0,2022-01-31,0,0,0,hold
1,2022-01-31,1,0,0,hold
2,2022-02-01,0,0,0,hold
3,2022-02-01,1,0,0,hold
4,2022-02-02,0,0,0,hold


In [21]:
# Export each trained action model as JSON so LEAN can load it inside Docker.
rl_model_paths = {}
for action_name, model in action_models.items():
    if model is None:
        continue

    artifact_path = STRATEGY_DIR / f"{config['model_name']}-q-{action_name}.json"
    model.save_model(artifact_path)
    rl_model_paths[action_name] = str(artifact_path)

# The metadata file is the contract between the research notebook and the LEAN
# strategy. It defines the feature order, action mapping, and indicator settings
# that must stay aligned at inference time.
metadata_path = STRATEGY_DIR / f"{config['model_name']}-policy-metadata.json"
metadata_path.write_text(
    json.dumps(
        {
            "model_name": config["model_name"],
            "policy_type": "fitted_q_iteration",
            "actions": {"hold": 0, "buy": 1, "sell": 2},
            "state_columns": STATE_COLUMNS,
            "gamma": GAMMA,
            "transaction_cost_bps": FEE_BPS,
            "indicator_parameters": {
                "macd_fast": 12,
                "macd_slow": 26,
                "macd_signal": 9,
                "rsi_period": 14,
                "cci_period": 20,
            },
            "gate_rules": {
                "buy": "macd_line crosses above macd_signal and rsi > 50 while flat",
                "sell": "macd_line crosses below macd_signal and rsi < 50 while long",
                "hold": "always valid",
            },
            "artifacts": rl_model_paths,
        },
        indent=2,
    )
)

print(f"Saved {len(rl_model_paths)} models + metadata -> {metadata_path.parent}/")

Saved 3 models + metadata -> /Users/renhao/git/github/RenQuant/backtesting/test_001_nvda/
